# Parkinson's MRI Classification — Colab Training Notebook
Run cells top to bottom. Set Runtime → GPU before starting.

In [ ]:
# 1. Clone your repo (once you've pushed it) OR just run standalone in Colab
!git clone https://github.com/YOUR_USERNAME/parkinsons-mri-classifier.git
%cd parkinsons-mri-classifier
!pip install -q -r requirements.txt

In [ ]:
# 2. Upload your kaggle.json (from kaggle.com -> Account -> Create New API Token)
from google.colab import files
uploaded = files.upload()  # select kaggle.json
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# 3. Download the dataset
!kaggle datasets download -d irfansheriff/parkinsons-brain-mri-dataset -p data --unzip
!find data -maxdepth 2 -type d
# If the folder names differ from 'normal'/'parkinson' after unzip, rename/move them here, e.g.:
# !mv data/<actual_folder>/normal data/normal
# !mv data/<actual_folder>/parkinson data/parkinson

In [ ]:
# 4. Sanity check the data loader
import sys
sys.path.append('src')
from dataset import build_balanced_filelist, stratified_split
fps, lbs = build_balanced_filelist('data', balance=True)
(Xtr, ytr), (Xv, yv), (Xte, yte) = stratified_split(fps, lbs)

In [ ]:
# 5. Train (EfficientNetB0 transfer learning)
!python src/train.py --data_dir data --epochs 25 --output model.h5

In [ ]:
# 6. Full evaluation: metrics, confusion matrix, ROC curve
!python src/evaluate.py --model model.h5

In [ ]:
# 7. Grad-CAM++ on a sample test image (pick any filepath from data/parkinson or data/normal)
import os
sample = os.path.join('data', 'parkinson', os.listdir('data/parkinson')[0])
!python src/gradcam.py --model model.h5 --image "{sample}" --output gradcam_output.png

from IPython.display import Image as IPImage
IPImage('gradcam_output.png')

In [ ]:
# 8. Download your trained model + result images before the Colab session ends
from google.colab import files
files.download('model.h5')
files.download('confusion_matrix.png')
files.download('roc_curve.png')
files.download('gradcam_output.png')

## Next steps
- Fill in the Results table in README.md with your actual numbers
- Commit code (not `model.h5` or `data/` — they're gitignored) and push to GitHub
- Optionally deploy `src/app.py` on Streamlit Community Cloud for a live demo link